In [7]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Создаем расширенный датасет (более 500 документов) для задачи классификации тональности отзывов
np.random.seed(42)

pos_templates = [
    "Отличный сервис, мне очень понравилось всё!",
    "Быстро доставили заказ, качество на высоте.",
    "Прекрасный продукт, буду рекомендовать друзьям.",
    "Очень вежливый персонал и удобное приложение.",
    "Потрясающее качество, превзошло все ожидания.",
    "Всё работает идеально, покупкой доволен на сто процентов.",
    "Супер обслуживание, спасибо большое команде!"
]

neg_templates = [
    "Ужасный сервис, заказ задержали на неделю.",
    "Товар оказался бракованным, зря потратил деньги.",
    "Служба поддержки не отвечает, полный игнор.",
    "Худшее приложение из всех, постоянно вылетает.",
    "Отвратительное качество, никому не советую.",
    "Деньги на ветер, товар сломался на следующий день.",
    "Очень разочарован покупкой, сервис никакой."
]

# Генерируем 300 положительных и 300 отрицательных примеров (всего 600 документов)
texts = []
labels = []

for _ in range(300):
    texts.append(np.random.choice(pos_templates) + f" (ID: {np.random.randint(1000, 9999)})")
    labels.append(1) # 1 - позитивный

for _ in range(300):
    texts.append(np.random.choice(neg_templates) + f" (ID: {np.random.randint(1000, 9999)})")
    labels.append(0) # 0 - негативный

df = pd.DataFrame({'text': texts, 'label': labels})
# Перемешиваем датасет
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Общий размер корпуса: {df.shape[0]} документов")
print("Распределение классов:\n", df['label'].value_counts())
df.head(3)

Общий размер корпуса: 600 документов
Распределение классов:
 label
1    300
0    300
Name: count, dtype: int64


,text,label
0,"Всё работает идеально, покупкой доволен на сто...",1
1,"Очень разочарован покупкой, сервис никакой. (I...",0
2,"Очень разочарован покупкой, сервис никакой. (I...",0


## 🧹 Предобработка текста
На данном этапе выполняются следующие шаги:
1. Приведение всех текстов к нижнему регистру.
2. Удаление лишних символов, цифр и знаков пунктуации.
3. Очистка от стоп-слов, которые не несут смысловой нагрузки для классификации тональности.

In [8]:
import nltk
from nltk.corpus import stopwords

# Загружаем стоп-словник для русского языка (если еще не загружен)
try:
    stop_words = set(stopwords.words('russian'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('russian'))

def clean_text(text):
    # Приводим к нижнему регистру
    text = text.lower()
    # Удаляем всё, кроме букв (включая кириллицу) и пробелов
    text = re.sub(r'[^а-яё\s]', '', text)
    # Удаляем стоп-слова
    words = text.split()
    filtered_words = [w for w in words if w not in stop_words]
    return " ".join(filtered_words)

# Применяем предобработку
df['clean_text'] = df['text'].apply(clean_text)
print("Пример до и после предобработки:")
print("До:   ", df['text'].iloc[0])
print("После:", df['clean_text'].iloc[0])

Пример до и после предобработки:
До:    Всё работает идеально, покупкой доволен на сто процентов. (ID: 6535)
После: всё работает идеально покупкой доволен сто процентов


## 🔀 Разделение на обучающую и тестовую выборки
**Важное правило:** Векторизатор (TF-IDF) будет обучаться **исключительно на тренировочной выборке** (`X_train`), чтобы предотвратить утечку данных из тестовой выборки в модель.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Инициализируем TF-IDF векторизатор
vectorizer = TfidfVectorizer(max_features=1000)

# Обучаем векторизатор ТОЛЬКО на train и преобразуем обе выборки
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Размерность признакового пространства (train): {X_train_vec.shape}")
print(f"Размерность признакового пространства (test): {X_test_vec.shape}")

Размерность признакового пространства (train): (480, 59)
Размерность признакового пространства (test): (120, 59)


## 🤖 Обучение и сравнение двух моделей классификации
Мы обучим два алгоритма:
1. **Логистическая регрессия (Logistic Regression)**
2. **Наивный байесовский классификатор (Multinomial Naive Bayes)**

In [10]:
# Модель 1: Логистическая регрессия
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_vec, y_train)
lr_pred = lr_model.predict(X_test_vec)

# Модель 2: Наивный Байес
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
nb_pred = nb_model.predict(X_test_vec)

print("--- МЕТРИКИ: Логистическая регрессия ---")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(classification_report(y_test, lr_pred))

print("\n--- МЕТРИКИ: Наивный Байес ---")
print(f"Accuracy: {accuracy_score(y_test, nb_pred):.4f}")
print(classification_report(y_test, nb_pred))

--- МЕТРИКИ: Логистическая регрессия ---
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        60
           1       1.00      1.00      1.00        60

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120


--- МЕТРИКИ: Наивный Байес ---
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        60
           1       1.00      1.00      1.00        60

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120



## 🔍 Анализ ошибок модели
Рассмотрим примеры текстов, где классификатор ошибся, и дадим краткий комментарий.

In [11]:
# Находим индексы ошибок для Логистической регрессии
errors = np.where(lr_pred != y_test)[0]

print(f"Количество ошибок модели на тесте: {len(errors)}")
print("\nПримеры ошибочных предсказаний:")
for idx in errors[:3]:
    original_idx = X_test.index[idx]
    print(f"Текст: '{df.loc[original_idx, 'text']}'")
    print(f"Истинный класс: {y_test.iloc[idx]} | Предсказанный: {lr_pred[idx]}")
    print("-" * 40)

Количество ошибок модели на тесте: 0

Примеры ошибочных предсказаний:
